# 05 · OD 분석 — 서울시 실측 OD ↔ 그래프 노드 연결

README '한계'의 **"실제 OD가 아닌 운행빈도 프록시 사용"** 을 해소하기 위한 모듈.
서울시 역간 OD(교통카드, 2026-07-22~28 7일치)와 부산·대구·대전 승하차 데이터를
이 저장소의 그래프 노드(`node_id = 노선번호|역번호`)에 연결한다.

`od_analysis.py` 의 함수를 그대로 호출한다 (로직이 두 곳으로 갈라지지 않도록).

> OD zip 7개(약 66MB)는 용량 문제로 저장소에 없다. `data/od/README.md` 안내대로
> 내려받아 `data/od/` 에 넣으면 전체가 실행되고, 없으면 매핑·지방 가중치만 산출된다.

In [1]:
# 저장소 루트에서 실행되도록 경로 이동 (notebooks/ 안에서 열었을 때 대비)
import os, sys
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())
print('작업 경로:', os.getcwd())

작업 경로: /tmp/claude-1002/-home-user02/0c7c1c23-b496-48b6-987f-820422a5be38/scratchpad/Raility


## 1. 역사마스터 ↔ 그래프 노드 매칭

역명 정규화(`norm_name`) + 호선 매핑(`LINE_MAP`) + 좌표 최근접 3단계.
이 저장소는 코레일 구간을 운영 노선(경부선·경인선·안산과천선 등)으로 나눠 두므로
서울시 마스터의 호선을 후보 노선 집합으로 넓게 매핑하고 좌표로 해소한다.

In [2]:
import pandas as pd
import od_analysis as oda

nodes = oda.load_nodes()
print('그래프 노드', len(nodes), '/ 수도권', (nodes.region == '수도권').sum())
mapping, unmatched, master = oda.build_mapping(nodes)
mapping.매칭방법.value_counts().to_frame('건수')

그래프 노드 1094 / 수도권 792


[mapping] master 784행 -> 매칭 774 / 미매칭 10 (매칭률 98.7%)


,건수
매칭방법,
name_line,770
coord_only,4


### 미매칭 내역 — 전부 그래프에 없는 노선(GTX-A)인지 확인

In [3]:
unmatched

,역사_ID,서울시_역명,서울시_호선,사유
0,9010,동탄,수도권 광역급행철도,그래프 미포함 노선(GTX-A)
1,9009,구성,수도권 광역급행철도,그래프 미포함 노선(GTX-A)
2,9008,성남,수도권 광역급행철도,그래프 미포함 노선(GTX-A)
3,9007,수서,수도권 광역급행철도,그래프 미포함 노선(GTX-A)
4,9006,삼성,수도권 광역급행철도,그래프 미포함 노선(GTX-A)
5,9005,서울,수도권 광역급행철도,그래프 미포함 노선(GTX-A)
6,9004,연신내,수도권 광역급행철도,그래프 미포함 노선(GTX-A)
7,9002,대곡,수도권 광역급행철도,그래프 미포함 노선(GTX-A)
8,9001,킨텍스,수도권 광역급행철도,그래프 미포함 노선(GTX-A)
9,9000,운정,수도권 광역급행철도,그래프 미포함 노선(GTX-A)


### 좌표로만 매칭된 역 (개명·표기 차이) — 좌표거리로 타당성 확인

In [4]:
mapping[mapping.매칭방법 != 'name_line']

,역사_ID,서울시_역명,서울시_호선,node_id,매칭방법,좌표거리_m
11,4814,원곡,서해선,I41WS|4814,coord_only,68.2
54,4512,운동장.송담대,에버라인선,L41E1|Y120,coord_only,14.8
270,2730,뚝섬유원지,7호선,S1107|0728,coord_only,2.2
591,1268,화전,경의중앙선,I4108|1268,coord_only,25.3


## 2. OD 7일치 집계 → 노드쌍 통행량

zip 이 있을 때만 실행 (**수 분 소요**). 지하철 역사 = ID 4자리 (9자리는 버스).

In [5]:
daily_avg = peak = stats = None
if len(oda.od_zips_available()) == len(oda.DAYS):
    od, raw_totals = oda.aggregate_od()
    daily_avg, peak, stats = oda.build_od_outputs(od, mapping)
else:
    print('OD zip 미배치 — data/od/README.md 안내대로 다운로드 후 재실행')

[od] 20260722: 지하철 OD쌍 290,753 / 통행 7,041,576 (주중)


[od] 20260723: 지하철 OD쌍 289,697 / 통행 7,008,714 (주중)


[od] 20260724: 지하철 OD쌍 294,931 / 통행 7,317,460 (주중)


[od] 20260725: 지하철 OD쌍 276,555 / 통행 5,069,555 (주말)


[od] 20260726: 지하철 OD쌍 255,586 / 통행 3,814,876 (주말)


[od] 20260727: 지하철 OD쌍 288,315 / 통행 6,716,805 (주중)


[od] 20260728: 지하철 OD쌍 290,569 / 통행 6,887,593 (주중)


[od] 총 통행 43,856,579 -> 매핑 후 43,259,076 (보존율 98.64%)


[out] od_daily_avg.csv: 381,261 노드쌍


[out] od_peak.csv: 269,706 노드쌍 (주중 5일 평균)


### 상위 통행쌍 — 상식 부합 점검 (강남·잠실·홍대 축이 최상위여야 정상)

In [6]:
if daily_avg is not None:
    lab = nodes.set_index('node_id')
    top = daily_avg[daily_avg.node_o != daily_avg.node_d].head(10).copy()
    top['출발'] = top.node_o.map(lab.역사명) + ' (' + top.node_o.map(lab.노선명) + ')'
    top['도착'] = top.node_d.map(lab.역사명) + ' (' + top.node_d.map(lab.노선명) + ')'
    display(top[['출발', '도착', 'trips_avg_daily',
                 'trips_avg_weekday', 'trips_avg_weekend']])
    sl = daily_avg[daily_avg.node_o == daily_avg.node_d]
    print(f'self-loop {len(sl):,}쌍, 통행 {sl.trips_avg_daily.sum():,.0f} '
          f'({sl.trips_avg_daily.sum() / daily_avg.trips_avg_daily.sum() * 100:.2f}%)'
          ' — 경로배정 시 제외 권장')

,출발,도착,trips_avg_daily,trips_avg_weekday,trips_avg_weekend
1,강남 (2호선),잠실(송파구청) (2호선),5222.57,5749.4,3905.5
2,잠실(송파구청) (2호선),강남 (2호선),5153.71,5631.2,3960.0
3,을지로입구 (2호선),홍대입구 (2호선),3818.00,3907.8,3593.5
4,강남 (2호선),신림 (2호선),3734.86,4301.2,2319.0
5,홍대입구 (2호선),을지로입구 (2호선),3693.86,3677.4,3735.0
6,신림 (2호선),강남 (2호선),3586.14,4052.6,2420.0
7,삼성(무역센터) (2호선),잠실(송파구청) (2호선),3437.00,3723.0,2722.0
8,서울대입구(관악구청) (2호선),강남 (2호선),3380.14,3854.8,2193.5
9,강남 (2호선),서울대입구(관악구청) (2호선),3351.57,3840.0,2130.5
10,삼성(무역센터) (2호선),강남 (2호선),3123.71,3499.0,2185.5


self-loop 735쌍, 통행 72,109 (1.17%) — 경로배정 시 제외 권장


## 3. 노드 가중치 — 서울 OD의 O합/D합 + 부산·대구·대전 승하차

`weight_source`: `seoul_od`(실측 OD) / `boarding_data`(지방 승하차) / `none`.

In [7]:
weights = oda.build_node_weights(nodes, daily_avg, mapping)
weights.weight_source.value_counts().to_frame('노드 수')

[out] node_weights.csv: 1094 노드, 커버리지 {'seoul_od': 763, 'boarding_data': 228, 'none': 103}


,노드 수
weight_source,
seoul_od,763
boarding_data,228
none,103


In [8]:
# 승차 상위 — 서울역·강남·잠실·홍대입구가 최상위여야 정상
weights[weights.weight_source == 'seoul_od'].nlargest(
    10, 'boarding_daily_avg')[['역사명', '노선명',
                              'boarding_daily_avg', 'alighting_daily_avg']]

,역사명,노선명,boarding_daily_avg,alighting_daily_avg
354,서울역,1호선,86802.22,79123.63
376,강남,2호선,80706.28,79270.12
370,잠실(송파구청),2호선,78846.43,77495.39
393,홍대입구,2호선,73731.70,79701.69
365,성수,2호선,57635.48,62220.88
386,구로디지털단지,2호선,54241.37,53895.19
384,신림,2호선,54031.56,53250.65
374,선릉,2호선,53715.17,47577.42
373,삼성(무역센터),2호선,52469.08,48538.66
418,고속터미널,3호선,52178.67,49777.73


In [9]:
# 지방(부산·대구·대전) 승차 상위 — 서면·반월당 등
weights[weights.weight_source == 'boarding_data'].nlargest(
    10, 'boarding_daily_avg')[['역사명', '노선명',
                              'boarding_daily_avg', 'alighting_daily_avg']]

,역사명,노선명,boarding_daily_avg,alighting_daily_avg
234,서면,부산 도시철도 1호선,36739.5,41718.8
268,서면,부산 도시철도 2호선,25602.9,25814.9
228,부산역,부산 도시철도 1호선,24770.0,25252.2
276,사상,부산 도시철도 2호선,23330.3,23219.1
143,반월당,대구 도시철도 2호선,22993.8,21131.9
255,센텀시티,부산 도시철도 2호선,21130.1,21020.6
217,하단,부산 도시철도 1호선,19618.8,18046.6
226,남포,부산 도시철도 1호선,18608.7,19493.4
114,동대구역,대구 도시철도 1호선,18534.7,18928.8
225,자갈치,부산 도시철도 1호선,18177.1,18360.6


### weight_source = none 인 노드 (노선별)

광주·부산김해경전철·동해선·대경선(승하차 데이터 미확보)과 서울시 마스터가
커버하지 않는 수도권 외곽 코레일 역.

In [10]:
(weights[weights.weight_source == 'none']
 .groupby(['region', '노선명']).size().to_frame('노드 수'))

노드 수
region 노선명                
광주     광주도시철도 1호선       20
대구     대경선               8
부산     동해선              23
       부산 경량도시철도 4호선     1
       부산 도시철도 3호선       1
       부산김해경전철          21
수도권    3호선               1
       경원선               3
       경의중앙선             3
       경춘선               1
       분당선               1
       서해선               6
       수인선               8
       자기부상철도            6

## 산출물 (data/od/processed/)

- `od_station_mapping.csv` / `od_station_unmatched.csv` / `node_weights.csv` — 커밋됨
- `od_daily_avg.csv` / `od_peak.csv` — 대용량이라 커밋 제외, 이 노트북(또는
  `python od_analysis.py`) 실행으로 재생성

analyze.py 연계 제안은 `docs/README_OD분석.md` 참조.